# [NOTEBOOK TEST CÔ LẬP] Dùng SD 1.5 gốc — KHÔNG dùng checkpoint chuyên biệt

**Mục đích duy nhất**: xác định checkpoint chuyên biệt (`specialized_unet_ckpt`)
có phải nguyên nhân gây lỗi tái tạo ảnh kiểu "cầu vồng/vector" hay không, bằng
cách chạy lại đúng luồng Inversion + Reconstruction nhưng ép `CKPT_DIR = None`
(dùng thẳng UNet gốc `runwayml/stable-diffusion-v1-5`, bỏ qua checkpoint đã
fine-tune).

**Cách đọc kết quả ở cell cuối cùng:**
- **ID Score ≥ 0.90** → checkpoint chuyên biệt trong `DATA_DIR` là nguyên nhân
  (sai/hỏng nội dung dù đúng dung lượng file) → cần tìm lại đúng checkpoint đã
  fine-tune thật, không dùng file hiện tại trong `DATA_DIR` nữa.
- **Vẫn cầu vồng/hỏng** → không phải do checkpoint → lỗi nằm trong chính thuật
  toán invert/reconstruct (nghi ngờ tiếp theo: `DualAttentionCapture` tự viết
  lại có thể không khớp 100% hành vi chuẩn của diffusers) → báo lại kết quả để
  đào sâu tiếp hướng đó.

**Lưu ý**: notebook này KHÔNG chạy batch FG-NET/FFHQ, không sinh kết quả để
báo cáo — chỉ để chẩn đoán. Chạy xong xác định được nguyên nhân thì quay lại
sửa đúng chỗ trong notebook pipeline chính (`FADING_pipeline_kaggle_3.ipynb`),
không dùng bản này để chạy full.

# Missing Person Search via FADING — Pipeline Suy diễn & Đánh giá (v3)

Chạy trên Kaggle Notebooks (GPU Tesla T4 / P100). Pipeline tối ưu cho độ phân giải chuẩn **512x512**, **50 bước DDIM**, can thiệp Dual-Attention (Self-Attention + Cross-Attention), khắc phục triệt để hiện tượng ám màu / tàn nhang bằng cách dùng UNet SD 1.5 nguyên bản, tối ưu Null-text Inversion không dừng sớm sai lệch và căn chỉnh ảnh đầu vào FFHQ chuẩn 512x512.


In [ ]:
# ===== KAGGLE: khong can mount gi ca - du lieu tu dong co san tai /kaggle/input =====
# In ra cau truc that de biet chinh xac ten thu muc dataset (phu thuoc ten ban dat
# luc tao dataset tren Kaggle, KHONG the biet truoc) - dung ket qua nay de dien vao
# cell "Khai bao duong dan" ben duoi.
import os

print("Cac dataset da gan vao notebook nay:")
for name in sorted(os.listdir("/kaggle/input")):
    print(f"  - {name}")

print("\nChi tiet 2 cap dau tien:")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level <= 2:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root) or root}/")
        if level == 2:
            for f in sorted(files)[:8]:
                print(f"{indent}  {f}")
            if len(files) > 8:
                print(f"{indent}  ... va {len(files)-8} file khac")


## Khai báo đường dẫn & Kiểm tra sẵn sàng trước khi chạy


In [ ]:
import os
import datetime
# ===== KAGGLE: SỬA đường dẫn dataset theo đúng tên dataset hiển thị ở cell trên =====
DATA_DIR = "/kaggle/input/datasets/menonkk/nckh-2025-2026"
# CKPT_DIR: Đường dẫn thư mục chứa checkpoint UNet đã fine-tune (512x512).
# Đặt CKPT_DIR = None nếu bạn muốn dùng trực tiếp UNet gốc của SD 1.5 (runwayml/stable-diffusion-v1-5).
CKPT_DIR = None  # === TEST CÔ LẬP: ép dùng thẳng SD 1.5 gốc, KHÔNG dùng checkpoint chuyên biệt ===
OUTPUT_DIR = "/kaggle/working/FADING_output"
FFHQ_DIR = os.path.join(DATA_DIR, "ffhq_aging_150_samples/ffhq_aging_150_samples")
LABELS_CSV = os.path.join(FFHQ_DIR, "sampled_labels.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# PREPROCESSED_FGNET_DIR: Đường dẫn thư mục ảnh FG-NET đã tiền xử lý offline (từ batch_preprocess_fgnet.ipynb)
# Có thể là Kaggle Dataset tải lên hoặc thư mục giải nén từ FGNET_preprocessed_full.zip
PREPROCESSED_FGNET_DIR = "/kaggle/input/fgnet-preprocessed-full/FGNET_preprocessed"
if not os.path.isdir(PREPROCESSED_FGNET_DIR):
    for alt_prep in [
        "/kaggle/working/FGNET_preprocessed",
        os.path.join(DATA_DIR, "FGNET_preprocessed"),
        os.path.join(OUTPUT_DIR, "FGNET_preprocessed"),
        r"d:\Data\project\nckh\data\FGNET_preprocessed"
    ]:
        if os.path.isdir(alt_prep):
            PREPROCESSED_FGNET_DIR = alt_prep
            break
print("=" * 70)
print("KIỂM TRA SẴN SÀNG TRƯỚC KHI CHẠY PIPELINE")
print("=" * 70)
all_ok = True
def check(label, condition, detail=""):
    global all_ok
    status = "✅" if condition else "❌"
    if not condition:
        all_ok = False
    print(f"{status} {label}" + (f"  — {detail}" if detail else ""))
# 1. GPU
import torch
has_gpu = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else "không có"
check("GPU khả dụng", has_gpu, gpu_name if has_gpu else "Vào Runtime > Change runtime type > T4 GPU")
# 2. Thư mục dữ liệu gốc
check("DATA_DIR tồn tại", os.path.isdir(DATA_DIR), DATA_DIR)
# 3. Thư mục 140 ảnh Specialization & labels CSV
ffhq_ok = os.path.isdir(FFHQ_DIR)
prep_ok = os.path.isdir(PREPROCESSED_FGNET_DIR)
check("Thư mục ảnh FG-NET đã tiền xử lý (PREPROCESSED_FGNET_DIR)", prep_ok, PREPROCESSED_FGNET_DIR if prep_ok else "Chưa tìm thấy -> Sẽ tự động dùng ảnh FG-NET gốc làm fallback")
check("Thư mục ffhq_aging_150_samples tồn tại", ffhq_ok)
if ffhq_ok:
    png_files = [f for f in os.listdir(FFHQ_DIR) if f.lower().endswith(".png")]
    check(f"Số ảnh .png trong đó (kỳ vọng ~140)", len(png_files) >= 100, f"đếm được {len(png_files)}")
    check("File sampled_labels.csv tồn tại", os.path.isfile(LABELS_CSV))
# 4. Ảnh test có trong 140 ảnh không
TEST_IMAGE_NAME = "01366.png"   # sửa nếu bạn chọn ảnh khác
if ffhq_ok:
    check(f"Ảnh test '{TEST_IMAGE_NAME}' có trong 140 ảnh", TEST_IMAGE_NAME in os.listdir(FFHQ_DIR))
# 5. Trạng thái checkpoint UNet
if CKPT_DIR is not None and os.path.isdir(CKPT_DIR):
    check(f"Sử dụng checkpoint UNet tùy chỉnh tại '{CKPT_DIR}'", True, "Checkpoint đã nạp")
else:
    check("Sử dụng UNet SD 1.5 nguyên bản", True, "runwayml/stable-diffusion-v1-5 (CKPT_DIR = None)")
print("=" * 70)
if all_ok:
    print("✅ TẤT CẢ ĐÃ SẴN SÀNG — có thể chạy tiếp các cell bên dưới.")
else:
    print("❌ CÒN THIẾU — xem lại các dòng ❌ ở trên trước khi chạy tiếp, tránh lỗi giữa chừng.")
print("=" * 70)


## Ghi log toàn bộ session ra file — để gửi cho Claude đọc

Từ đây trở đi, MỌI dòng `print()` (kể cả log của các cell sau) sẽ vừa hiện trên
màn hình như bình thường, vừa được ghi thêm vào 1 file `.txt` trong OUTPUT_DIR — không
cần copy tay từng đoạn log nữa, chỉ cần gửi file này.

Dùng chế độ **append** (nối thêm, không ghi đè) — nếu phiên Kaggle bị ngắt rồi mở
lại, chạy lại đúng cell này, log cũ vẫn còn nguyên, log mới nối tiếp vào sau.

In [ ]:
import sys
import datetime

LOG_FILE_PATH = os.path.join(OUTPUT_DIR, "full_session_log.txt")

# SUA (lan 2): "isinstance(sys.stdout, TeeLogger)" KHONG dang tin - moi lan cell
# nay chay lai, Python tao ra 1 class TeeLogger MOI (du code giong het), nen
# isinstance luon that bai, cu the boc them 1 lop moi -> boc chong vo han lan,
# gay loi de quy AttributeError (thay ro trong traceback: isatty() goi lap 4 lan).
#
# Cach sua dung tin cay: "boc tach" theo kieu duck-typing (kiem tra co thuoc tinh
# .terminal hay khong, khong quan tam class nao), lan xuong TAN GOC stdout that su
# (khong con thuoc tinh .terminal nua) - bat ke da bi boc chong bao nhieu lop truoc do.
_true_stdout = sys.stdout
while hasattr(_true_stdout, "terminal"):
    _true_stdout = _true_stdout.terminal

class TeeLogger:
    """Ghi dong thoi ra man hinh (terminal) VA ra file - khong mat cai nao ca."""
    def __init__(self, filepath, mode="a"):
        self.terminal = _true_stdout   # LUON tro thang ve stdout GOC, khong bao gio boc chong
        self.log_file = open(filepath, mode, encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.log_file.flush()

    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

    def isatty(self):
        return self.terminal.isatty()

    def __getattr__(self, name):
        return getattr(self.terminal, name)

sys.stdout = TeeLogger(LOG_FILE_PATH, mode="a")

print(f"\n{'='*70}")
print(f"===== BẮT ĐẦU / TIẾP TỤC GHI LOG — {datetime.datetime.now()} =====")
print(f"===== File log: {LOG_FILE_PATH} =====")
print(f"{'='*70}\n")


## Cài đặt thư viện

Colab đã có sẵn `torch`, `numpy`, `pandas`, `pillow`, `opencv-python`, `tqdm` — chỉ cần cài thêm phần chưa có.

In [ ]:
# ===== 1. Cài đặt các thư viện cơ bản & mô hình =====
!pip install -q diffusers transformers accelerate bitsandbytes insightface onnxruntime-gpu huggingface_hub
!pip install -q "git+https://github.com/WildChlamydia/MiVOLO.git"
!pip install -q basicsr facexlib gfpgan

# ===== 2. Chuẩn bị CodeFormer cho pipeline tiền xử lý ảnh FG-NET =====
import os
import subprocess
import sys

CODEFORMER_DIR = "/kaggle/working/CodeFormer"
if not os.path.exists(CODEFORMER_DIR):
    print("Đang clone CodeFormer từ GitHub...")
    !git clone https://github.com/sczhou/CodeFormer.git {CODEFORMER_DIR}
    %cd {CODEFORMER_DIR}
    !pip install -q -r requirements.txt
    !python basicsr/setup.py develop
    %cd /kaggle/working
    
    # Tải weights facelib và CodeFormer
    print("Đang tải pretrained weights cho CodeFormer...")
    %cd {CODEFORMER_DIR}
    !python scripts/download_pretrained_models.py facelib
    !python scripts/download_pretrained_models.py CodeFormer
    %cd /kaggle/working
    print("✅ Đã tải xong toàn bộ weights của CodeFormer.")
else:
    print("Thư mục CodeFormer đã tồn tại.")

# ===== 3. Vá lỗi torchvision.transforms.functional_tensor trong basicsr =====
res = subprocess.run([sys.executable, "-m", "pip", "show", "basicsr"], capture_output=True, text=True)
site_packages_dir = None
for line in res.stdout.splitlines():
    if line.startswith("Location:"):
        site_packages_dir = line.split("Location:")[1].strip()
        break

files_to_patch = []
if site_packages_dir:
    files_to_patch.append(os.path.join(site_packages_dir, "basicsr", "data", "degradations.py"))
files_to_patch.append(os.path.join(CODEFORMER_DIR, "basicsr", "data", "degradations.py"))

patched_count = 0
for deg_file in files_to_patch:
    if os.path.exists(deg_file):
        with open(deg_file, "r", encoding="utf-8") as f:
            content = f.read()
        if "functional_tensor" in content:
            content = content.replace(
                "from torchvision.transforms.functional_tensor import",
                "from torchvision.transforms.functional import"
            )
            content = content.replace("functional_tensor", "functional")
            with open(deg_file, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"✅ Đã vá lỗi functional_tensor tại: {deg_file}")
            patched_count += 1
        else:
            print(f"ℹ️ File {deg_file} đã chuẩn (không chứa functional_tensor).")

print(f"\nHoàn tất chuẩn bị môi trường & CodeFormer. Số file đã vá: {patched_count}")


## Cấu hình pipeline (hyperparameters) — tương đương `configs/config.yaml`

In [ ]:
config = {
    "base_model": {
        "pretrained_model_name_or_path": "runwayml/stable-diffusion-v1-5",
    },
    "inversion": {
        "num_inference_steps": 50,
        "guidance_scale": 7.5,           # SỬA: revert về 7.5 (giá trị gốc đã xác nhận đúng qua
                                          # review code chính thức FADING) — bản 4.0 trước đó làm
                                          # sanity-check tái tạo (reconstruction) bị lệch cấu trúc
                                          # nghiêm trọng (ID Score ~0.23, kiểu lỗi "vector/hoạt hình"
                                          # đã từng gặp). Null-text Inversion tối ưu null_embeddings
                                          # RIÊNG cho 1 giá trị guidance_scale cố định — đổi giá trị
                                          # này phá vỡ toàn bộ cân bằng đã tối ưu.
        "num_inner_steps": 15,           # Tăng lên 15 để tối ưu hóa vector null-text triệt để
        "early_stop_epsilon": 1e-5,
        "image_size": 512,
    },
    "editing": {
        "guidance_scale": 7.5,           # SỬA: revert về 7.5, BẮT BUỘC khớp với
                                          # config["inversion"]["guidance_scale"]. Lý do: null_embeddings
                                          # được tối ưu bởi Inverter dùng 1 guidance_scale cố định — Editor
                                          # (reconstruct() VÀ edit()) tái sử dụng CHÍNH null_embeddings đó
                                          # để tính CFG, nên guidance_scale 2 bên PHẢI giống hệt nhau, nếu
                                          # không sẽ phá vỡ toàn bộ phép tối ưu null-text (đã xác nhận đây
                                          # là nguyên nhân thật của lỗi tái tạo "vector/hoạt hình", ID Score
                                          # ~0.23-0.28, không phải do thiếu align hay num_inner_steps thấp).
                                          # Nếu vấn đề "cháy màu/tàn nhang" ở tuổi cực đoan có thật, cần sửa
                                          # bằng cơ chế KHÁC (vd: attention_control_ratio, hậu xử lý màu),
                                          # không phải tách rời guidance_scale giữa Inversion và Editing.
        "attention_control_ratio": 0.8,
        "image_size": 512,
    },
    "embedding": {
        "model_name": "buffalo_l",
        "ctx_id": -1,
        "det_size": (256, 256),
    },
    "paths": {
        "ffhq_dir": FFHQ_DIR,
        "labels_csv": LABELS_CSV,
        "specialized_unet_ckpt": CKPT_DIR,
        "output_dir": os.path.join(OUTPUT_DIR, "edited_images"),
    },
}
config


## Hàm tiện ích dùng chung & Prompt Helpers (tương đương `src/utils/prompts.py`)


In [ ]:
from typing import Dict
# Trung diem tung age_group trong sampled_labels.csv.
# Rieng nhom cuoi "70-120" LAY TAY = 80, khong dung trung diem toan hoc (se ra 95),
# vi paper FADING goc ghi ro: "For the oldest age group (70+), we translate to 80 years old".
AGE_GROUP_TO_AGE: Dict[str, int] = {
    "0-2": 1,
    "3-6": 4,
    "7-9": 8,
    "10-14": 12,
    "15-19": 17,
    "20-29": 24,
    "30-39": 34,
    "40-49": 44,
    "50-69": 59,
    "70-120": 80,
}
def age_group_to_age(age_group: str) -> int:
    """Quy doi 1 nhan age_group (vd "30-39") sang 1 con so tuoi dai dien (vd 34)."""
    return AGE_GROUP_TO_AGE[age_group]
def gender_to_word(gender: str, age: int) -> str:
    """Quy doi gender ("male"/"female") + tuoi sang tu mo ta gioi tinh dung trong prompt:
    woman/man cho nguoi lon (age >= 15), girl/boy neu age < 15."""
    is_female = gender.lower() == "female"
    if age < 15:
        return "girl" if is_female else "boy"
    return "woman" if is_female else "man"
def build_prompt_alpha(age: int, gender_word: str) -> str:
    """Build P_alpha = "photo of a {age} year old {gender_word}"."""
    return f"photo of a {age} year old {gender_word}"
def build_prompt_neutral(gender_word: str) -> str:
    """Build P_neutral = "photo of a {gender_word}" - prompt trung lap, khong chua tuoi."""
    return f"photo of a {gender_word}"
def build_prompt_tau(target_age: int, gender_word: str) -> str:
    """Build P_tau = "photo of a {target_age} year old {gender_word}".
    SỬA: đã revert bỏ "Enhanced Prompts" (từ khóa da theo tuổi) — kỹ thuật này đã được
    ablation-test độc lập từ trước và XÁC NHẬN làm điểm ID Score TỆ HƠN ~46% so với
    prompt đơn giản, quyết định KHÔNG dùng. Bản có Enhanced Prompts bị đưa lại vào code
    là hồi quy, không phải cải tiến — giữ prompt đơn giản đúng bản đã kiểm chứng."""
    return f"photo of a {target_age} year old {gender_word}"


## Module 2 — Null-text Inversion (tương đương `src/fading/inversion.py`)

In [ ]:
from typing import Dict as _Dict, Optional, List, Tuple
import os
import torch
from torchvision import transforms
import torch.nn.functional as F
from torch.optim import Adam
from PIL import Image
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
NUM_DDIM_STEPS_DEFAULT = 50
GUIDANCE_SCALE_DEFAULT = 4.0
# Chi luu attention map cua layer co do phan giai <= 32x32 - giam VRAM, dung theo toi uu cua
# Prompt-to-Prompt goc (layer do phan giai thap mang nhieu ngu nghia hon).
MAX_ATTN_RESOLUTION = 32 * 32
class DualAttentionCapture:
    """
    Bắt đồng thời Self-Attention (attn1) và Cross-Attention (attn2)
    trong quá trình DDIM Inversion.
    """
    def __init__(self, unet: UNet2DConditionModel):
        self.unet = unet
        self._orig_processors = unet.attn_processors
        self.captured_self: _Dict[str, torch.Tensor] = {}
        self.captured_cross: _Dict[str, torch.Tensor] = {}
        self.enabled = False
    def _build_processor(self, name: str):
        capture = self
        is_cross = name.endswith("attn2.processor")
        class _CapturingProcessor:
            def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, **kwargs):
                query = attn.to_q(hidden_states)
                context = encoder_hidden_states if encoder_hidden_states is not None else hidden_states
                key = attn.to_k(context)
                value = attn.to_v(context)
                query = attn.head_to_batch_dim(query)
                key = attn.head_to_batch_dim(key)
                value = attn.head_to_batch_dim(value)
                attention_probs = attn.get_attention_scores(query, key, attention_mask)
                # Chỉ lưu attention map của các layer có số lượng spatial tokens <= 32x32
                if capture.enabled and attention_probs.shape[1] <= MAX_ATTN_RESOLUTION:
                    if is_cross:
                        capture.captured_cross[name] = attention_probs.detach().cpu()
                    else:
                        capture.captured_self[name] = attention_probs.detach().cpu()
                hidden_states = torch.bmm(attention_probs, value)
                hidden_states = attn.batch_to_head_dim(hidden_states)
                hidden_states = attn.to_out[0](hidden_states)
                hidden_states = attn.to_out[1](hidden_states)
                return hidden_states
        return _CapturingProcessor()
    def register(self) -> None:
        # Gắn processor vào toàn bộ các layer attention (cả attn1 và attn2)
        new_processors = {name: self._build_processor(name) for name in self.unet.attn_processors.keys()}
        self.unet.set_attn_processor(new_processors)
    def restore(self) -> None:
        self.unet.set_attn_processor(self._orig_processors)
    def capture_step(self, forward_fn):
        self.captured_self = {}
        self.captured_cross = {}
        self.enabled = True
        with torch.no_grad():
            result = forward_fn()
        self.enabled = False
        return dict(self.captured_self), dict(self.captured_cross), result
class NullTextInverter:
    def __init__(
        self,
        pretrained_model_name_or_path: str = "runwayml/stable-diffusion-v1-5",
        unet_checkpoint_dir: Optional[str] = None,
        device: str = "cuda",
        num_inference_steps: int = 50,         # Đưa về 50 bước chuẩn
        guidance_scale: float = 7.5,
        num_inner_steps: int = 10,
        early_stop_epsilon: float = 1e-5,
        image_size: int = 512,                 # Đưa lên 512x512
    ):
        self.pretrained_model_name_or_path = pretrained_model_name_or_path
        self.unet_checkpoint_dir = unet_checkpoint_dir
        self.device = device
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.num_inner_steps = num_inner_steps
        self.early_stop_epsilon = early_stop_epsilon
        self.image_size = image_size
        self.vae = None
        self.unet = None
        self.text_encoder = None
        self.tokenizer = None
        self.scheduler = None
    def _load_models(self) -> None:
        model_id = self.pretrained_model_name_or_path
        self.vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float16)
        self.text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.float16)
        self.tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
        if self.unet_checkpoint_dir and os.path.isdir(self.unet_checkpoint_dir):
            self.unet = UNet2DConditionModel.from_pretrained(self.unet_checkpoint_dir, torch_dtype=torch.float16)
        else:
            self.unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=torch.float16)
        self.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")
        self.scheduler.set_timesteps(self.num_inference_steps)
        self.vae.to(self.device).eval().requires_grad_(False)
        self.text_encoder.to(self.device).eval().requires_grad_(False)
        self.unet.to(self.device).eval().requires_grad_(False)
    def _load_image_latent(self, image_path: str) -> torch.Tensor:
        transform = transforms.Compose([
            transforms.Resize((self.image_size, self.image_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        image = Image.open(image_path).convert("RGB")
        image_t = transform(image).unsqueeze(0).to(self.device, dtype=torch.float16)
        with torch.no_grad():
            latent = self.vae.encode(image_t).latent_dist.mean * self.vae.config.scaling_factor
        return latent
    def _encode_text(self, prompt: str) -> torch.Tensor:
        tokens = self.tokenizer(
            [prompt],
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
            return self.text_encoder(tokens.input_ids)[0]
    def _predict_noise(self, latent: torch.Tensor, t, embedding: torch.Tensor) -> torch.Tensor:
        return self.unet(latent, t, encoder_hidden_states=embedding).sample
    def _ddim_next_step(self, noise_pred: torch.Tensor, t: int, sample: torch.Tensor) -> torch.Tensor:
        step = self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        timestep, next_timestep = min(t - step, 999), t
        alpha_prod_t = self.scheduler.alphas_cumprod[timestep] if timestep >= 0 else self.scheduler.final_alpha_cumprod
        alpha_prod_t_next = self.scheduler.alphas_cumprod[next_timestep]
        beta_prod_t = 1 - alpha_prod_t
        next_original_sample = (sample - beta_prod_t**0.5 * noise_pred) / alpha_prod_t**0.5
        next_sample_direction = (1 - alpha_prod_t_next)**0.5 * noise_pred
        return alpha_prod_t_next**0.5 * next_original_sample + next_sample_direction
    def _ddim_prev_step(self, noise_pred: torch.Tensor, t: int, sample: torch.Tensor) -> torch.Tensor:
        step = self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        prev_timestep = t - step
        alpha_prod_t = self.scheduler.alphas_cumprod[t]
        alpha_prod_t_prev = self.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else self.scheduler.final_alpha_cumprod
        beta_prod_t = 1 - alpha_prod_t
        pred_original_sample = (sample - beta_prod_t**0.5 * noise_pred) / alpha_prod_t**0.5
        pred_sample_direction = (1 - alpha_prod_t_prev)**0.5 * noise_pred
        return alpha_prod_t_prev**0.5 * pred_original_sample + pred_sample_direction
    def _ddim_inversion(self, z0: torch.Tensor, cond_embedding: torch.Tensor) -> List[torch.Tensor]:
        latent = z0.clone().detach()
        pivot_latents = [latent]
        timesteps = self.scheduler.timesteps
        for i in range(self.num_inference_steps):
            t = timesteps[len(timesteps) - i - 1]
            with torch.no_grad():
                noise_pred = self._predict_noise(latent, t, cond_embedding)
                latent = self._ddim_next_step(noise_pred, t, latent)
            pivot_latents.append(latent)
        return pivot_latents
    def _null_text_optimization(
        self,
        pivot_latents: List[torch.Tensor],
        uncond_embedding: torch.Tensor,
        cond_embedding: torch.Tensor,
        attn_capture: DualAttentionCapture,
    ):
        uncond_embeddings = uncond_embedding.clone()
        null_embeddings_list: List[torch.Tensor] = []
        self_attention_maps: _Dict[int, _Dict[str, torch.Tensor]] = {}
        cross_attention_maps: _Dict[int, _Dict[str, torch.Tensor]] = {}
        latent_cur = pivot_latents[-1]
        timesteps = self.scheduler.timesteps
        for i in range(self.num_inference_steps):
            uncond_embeddings = uncond_embeddings.clone().detach().float().requires_grad_(True)
            # Giữ LR = 1e-2 cho 25 bước đầu, giảm dần ở các bước sau
            lr_scale = 1.0 if i < 25 else max(0.4, 1.0 - (i - 25) / 35.0)
            optimizer = Adam([uncond_embeddings], lr=1e-2 * lr_scale)
            latent_prev = pivot_latents[len(pivot_latents) - i - 2]
            t = timesteps[i]
            with torch.no_grad():
                noise_pred_cond = self._predict_noise(latent_cur, t, cond_embedding)
            for inner_step in range(self.num_inner_steps):
                noise_pred_uncond = self._predict_noise(latent_cur, t, uncond_embeddings.half())
                noise_pred = noise_pred_uncond + self.guidance_scale * (noise_pred_cond - noise_pred_uncond)
                latent_prev_rec = self._ddim_prev_step(noise_pred, t, latent_cur)
                loss = F.mse_loss(latent_prev_rec, latent_prev)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                # Dừng sớm CHỈ khi loss thực sự nhỏ, KHÔNG cộng dồn i * 2e-5
                if loss.item() < self.early_stop_epsilon:
                    break
            null_embeddings_list.append(uncond_embeddings[:1].detach().half())
            with torch.no_grad():
                noise_pred_uncond_final = self._predict_noise(latent_cur, t, uncond_embeddings.half())
            # Bắt đồng thời cả Self và Cross Attention maps tại bước CFG thực
            s_maps, c_maps, noise_pred_cond_final = attn_capture.capture_step(
                lambda: self._predict_noise(latent_cur, t, cond_embedding)
            )
            self_attention_maps[int(t)] = s_maps
            cross_attention_maps[int(t)] = c_maps
            with torch.no_grad():
                noise_pred = noise_pred_uncond_final + self.guidance_scale * (
                    noise_pred_cond_final - noise_pred_uncond_final
                )
                latent_cur = self._ddim_prev_step(noise_pred, t, latent_cur)
            print(f"[NullTextInverter] t={int(t)} ({i + 1}/{self.num_inference_steps}) loss={loss.item():.6f}")
        return null_embeddings_list, (self_attention_maps, cross_attention_maps)
    def invert(self, image_path: str, initial_age: int, gender_word: str):
        if self.unet is None:
            self._load_models()
        p_alpha = build_prompt_alpha(initial_age, gender_word)
        uncond_embedding = self._encode_text("")
        cond_embedding = self._encode_text(p_alpha)
        z0 = self._load_image_latent(image_path)
        pivot_latents = self._ddim_inversion(z0, cond_embedding)
        # --- TỐI ƯU HÓA CHO TRẺ EM / TODDLER (initial_age < 10) ---
        # Trẻ em có cấu trúc xương mặt thay đổi mạnh, cần tăng số bước inner_steps 
        # trong Null-text optimization để bám sát đặc trưng không gian tốt hơn.
        original_inner_steps = self.num_inner_steps
        if initial_age < 10:
            self.num_inner_steps = max(self.num_inner_steps, 20)  # Tăng lên 20 bước cho trẻ nhỏ
            print(f"[NullTextInverter] Phát hiện độ tuổi trẻ em ({initial_age} tuổi) -> Tăng num_inner_steps lên {self.num_inner_steps} để tối ưu cấu trúc xương mặt.")
        else:
            print(f"[NullTextInverter] Độ tuổi người lớn/thiếu niên ({initial_age} tuổi) -> Dùng num_inner_steps chuẩn: {self.num_inner_steps}")
        attn_capture = DualAttentionCapture(self.unet)
        attn_capture.register()
        try:
            null_embeddings_list, (self_maps, cross_maps) = self._null_text_optimization(
                pivot_latents, uncond_embedding, cond_embedding, attn_capture
            )
        finally:
            attn_capture.restore()
            # Khôi phục lại giá trị gốc sau khi chạy xong
            self.num_inner_steps = original_inner_steps
        z_T = pivot_latents[-1]
        return z_T, null_embeddings_list, (self_maps, cross_maps)


## Module 3 — Editing (tương đương `src/fading/editing.py`)

In [ ]:
from typing import Dict as _Dict, Optional, List, Tuple
import os
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer


# SUA: Thêm hàm get_word_inds phục vụ cơ chế LocalBlend (Prompt-to-Prompt - Hertz et al.)
def get_word_inds(prompt: str, word: str, tokenizer) -> np.ndarray:
    """
    Tìm vị trí token của từ `word` trong `prompt` đã tokenize bởi CLIPTokenizer.
    Dùng để định vị token chủ thể chung (như "person", "woman", "man") giữa prompt gốc và prompt đích.
    Trả về mảng 1D các index token.
    """
    word_clean = word.strip().lower()
    tokens = tokenizer.encode(prompt)
    inds = []
    for idx, token_id in enumerate(tokens):
        tok_str = tokenizer.decode([token_id]).strip().lower().replace("</w>", "").strip(",.!?\"'")
        if tok_str and (tok_str == word_clean or word_clean in tok_str):
            inds.append(idx)
    if not inds:
        # Fallback nếu từ bị tách nhỏ qua subwords
        for idx, token_id in enumerate(tokens):
            tok_str = tokenizer.decode([token_id]).strip().lower()
            if word_clean in tok_str or (len(tok_str) > 2 and tok_str in word_clean):
                inds.append(idx)
    if not inds:
        # Fallback an toàn cuối cùng: token thứ 4 (thường là danh từ sau "photo of a")
        inds = [4]
    return np.array(inds, dtype=int)


# SUA: Triển khai công thức LocalBlend từ Prompt-to-Prompt gốc (pipeline_prompt2prompt.py)
def local_blend(
    recon_latent: torch.Tensor,
    edit_latent: torch.Tensor,
    cross_attn_maps_recon: List[torch.Tensor],
    cross_attn_maps_edit: List[torch.Tensor],
    word_inds_recon: np.ndarray,
    word_inds_edit: np.ndarray,
    threshold: float = 0.3,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Cơ chế LocalBlend (Hertz et al. - Prompt-to-Prompt):
    Chỉ cho phép thay đổi ở ĐÚNG vùng chứa chủ thể (face/person), GIỮ NGUYÊN latent gốc
    ở mọi vùng khác (nền, tóc, quần áo) tại MỖI bước denoising (không chỉ hậu xử lý 1 lần).

    Công thức gốc:
        maps = torch.cat([cross_attn_maps_recon, cross_attn_maps_edit], dim=0)
        maps = (maps * alpha_layers).sum(-1).mean(...)
        mask = F.max_pool2d(maps, 3, 1, padding=1)
        mask = F.interpolate(mask, size=recon_latent.shape[2:])
        mask = mask / mask.max(...)
        mask = mask.gt(threshold)
        mask = (mask[:1] | mask[1:]).to(dtype=edit_latent.dtype)
        return recon_latent + mask * (edit_latent - recon_latent)
    """
    if len(cross_attn_maps_recon) == 0 or len(cross_attn_maps_edit) == 0:
        return edit_latent, torch.ones_like(edit_latent[:, :1])

    if len(word_inds_recon) == 0:
        word_inds_recon = np.array([4])
    if len(word_inds_edit) == 0:
        word_inds_edit = np.array([4])

    # 1. Trích xuất cross-attention maps của các token chủ thể cho recon và edit
    # Mỗi map có shape [num_heads, 256, 77] (với 256 = 16x16)
    recon_maps = []
    for m in cross_attn_maps_recon:
        sub_m = m[:, :, word_inds_recon].mean(dim=-1).reshape(-1, 16, 16)
        recon_maps.append(sub_m)

    edit_maps = []
    for m in cross_attn_maps_edit:
        sub_m = m[:, :, word_inds_edit].mean(dim=-1).reshape(-1, 16, 16)
        edit_maps.append(sub_m)

    # 2. Gom các layer và tính trung bình theo heads và layers -> [1, 1, 16, 16]
    recon_avg = torch.cat(recon_maps, dim=0).mean(dim=0, keepdim=True).unsqueeze(0)
    edit_avg = torch.cat(edit_maps, dim=0).mean(dim=0, keepdim=True).unsqueeze(0)

    # 3. Ghép recon và edit: shape [2, 1, 16, 16]
    maps = torch.cat([recon_avg, edit_avg], dim=0)

    # 4. Max-pool 3x3 để mở rộng bao phủ biên chủ thể
    maps = F.max_pool2d(maps, kernel_size=3, stride=1, padding=1)

    # 5. Nội suy lên kích thước latent (64x64)
    maps = F.interpolate(maps, size=recon_latent.shape[2:], mode="bilinear", align_corners=False)

    # 6. Chuẩn hóa về [0, 1] cho mỗi map
    max_val = maps.flatten(2).max(dim=-1)[0].unsqueeze(-1).unsqueeze(-1).clamp(min=1e-8)
    norm_maps = maps / max_val

    # 7. Nhị phân hóa với ngưỡng threshold
    mask = norm_maps.gt(threshold)

    # 8. Hợp nhất: vùng chủ thể ở recon HOẶC ở edit -> shape [1, 1, H, W]
    # Ép kiểu mask theo đúng dtype của edit_latent (float16/float32) để tránh lỗi lệch dtype với UNet
    mask = (mask[:1] | mask[1:]).to(dtype=edit_latent.dtype)

    # 9. Pha trộn latent: recon_latent + mask * (edit_latent - recon_latent)
    blended = recon_latent + mask * (edit_latent - recon_latent)
    return blended, mask


class DualAttentionInjector:
    """
    Tiêm Self-Attention (attn1) để khóa hình học khuôn mặt,
    và tiêm Cross-Attention (attn2) có chọn lọc để hòa trộn tuổi tác mà không làm méo mặt.
    Bổ sung khả năng bắt live cross-attention maps của các layer 16x16 phục vụ LocalBlend.
    """
    def __init__(
        self,
        unet: UNet2DConditionModel,
        self_maps: _Dict[int, _Dict[str, torch.Tensor]],
        cross_maps: _Dict[int, _Dict[str, torch.Tensor]],
        attention_control_ratio: float,
        num_inference_steps: int,
    ):
        self.unet = unet
        self._orig_processors = unet.attn_processors
        self.self_maps = self_maps
        self.cross_maps = cross_maps
        self.attention_control_ratio = attention_control_ratio
        self.num_inference_steps = num_inference_steps
        self.enabled = False
        self.current_t: Optional[int] = None
        # SUA: Thêm trạng thái bắt live cross-attention cho LocalBlend
        self.capture_mode: Optional[str] = None  # "recon", "edit", hoặc None
        self.captured_cross: _Dict[str, List[torch.Tensor]] = {"recon": [], "edit": []}

    def _build_processor(self, name: str):
        injector = self
        is_cross = name.endswith("attn2.processor")
        class _InjectingProcessor:
            def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, **kwargs):
                query = attn.to_q(hidden_states)
                context = encoder_hidden_states if encoder_hidden_states is not None else hidden_states
                key = attn.to_k(context)
                value = attn.to_v(context)  # Value vector luôn tính từ prompt mới P_tau
                query = attn.head_to_batch_dim(query)
                key = attn.head_to_batch_dim(key)
                value = attn.head_to_batch_dim(value)
                attention_probs = attn.get_attention_scores(query, key, attention_mask)

                # SUA: bắt attention TRƯỚC khi bị ghi đè, đảm bảo LocalBlend dùng dữ liệu sống
                if injector.capture_mode is not None and is_cross and attention_probs.shape[1] == 256:
                    injector.captured_cross[injector.capture_mode].append(attention_probs.detach())

                if injector.enabled:
                    if not is_cross:
                        # 1. Khóa hình học khuôn mặt bằng Self-Attention (attn1)
                        ref_self = injector.self_maps.get(injector.current_t, {}).get(name)
                        if ref_self is not None:
                            attention_probs = ref_self.to(device=value.device, dtype=value.dtype)
                    else:
                        # 2. Tiêm Cross-Attention (attn2):
                        # Giữ các token chung: "<start>", "photo", "of", "a" (index 0..3)
                        # và các token đệm/EOS phía sau (index >= 7).
                        # Thả tự do các token tuổi (index 4..6: "{age}", "year", "old") để nếp nhăn sinh tự nhiên.
                        ref_cross = injector.cross_maps.get(injector.current_t, {}).get(name)
                        if ref_cross is not None:
                            ref_cross = ref_cross.to(device=value.device, dtype=value.dtype)
                            if attention_probs.shape[-1] == ref_cross.shape[-1]:
                                attention_probs[:, :, :4] = ref_cross[:, :, :4]
                                attention_probs[:, :, 7:] = ref_cross[:, :, 7:]

                hidden_states = torch.bmm(attention_probs, value)
                hidden_states = attn.batch_to_head_dim(hidden_states)
                hidden_states = attn.to_out[0](hidden_states)
                hidden_states = attn.to_out[1](hidden_states)
                return hidden_states
        return _InjectingProcessor()

    def register(self) -> None:
        # Gắn processor vào toàn bộ các layer attention (cả attn1 và attn2)
        new_processors = {name: self._build_processor(name) for name in self.unet.attn_processors.keys()}
        self.unet.set_attn_processor(new_processors)

    def restore(self) -> None:
        self.unet.set_attn_processor(self._orig_processors)

    def inject_step(self, step_index: int, t, forward_fn):
        self.current_t = int(t)
        self.enabled = step_index < (self.attention_control_ratio * self.num_inference_steps)
        with torch.no_grad():
            result = forward_fn()
        self.enabled = False
        return result


class Editor:
    def __init__(
        self,
        pretrained_model_name_or_path: str = "runwayml/stable-diffusion-v1-5",
        unet_checkpoint_dir: Optional[str] = None,
        device: str = "cuda",
        num_inference_steps: int = 50,
        guidance_scale: float = 7.5,
        attention_control_ratio: float = 0.8,
        image_size: int = 512,
        use_local_blend: bool = False,               # SUA: Mặc định TẮT LocalBlend để không đổi hành vi mặc định
        local_blend_threshold: float = 0.3,          # SUA: Ngưỡng binarize 0.3 chuẩn Hertz et al.
    ):
        self.pretrained_model_name_or_path = pretrained_model_name_or_path
        self.unet_checkpoint_dir = unet_checkpoint_dir
        self.device = device
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.attention_control_ratio = attention_control_ratio
        self.image_size = image_size
        self.use_local_blend = use_local_blend
        self.local_blend_threshold = local_blend_threshold
        self.vae = None
        self.unet = None
        self.text_encoder = None
        self.tokenizer = None
        self.scheduler = None

    def _load_models(self) -> None:
        model_id = self.pretrained_model_name_or_path
        self.vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float16)
        self.text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.float16)
        self.tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
        if self.unet_checkpoint_dir and os.path.isdir(self.unet_checkpoint_dir):
            self.unet = UNet2DConditionModel.from_pretrained(self.unet_checkpoint_dir, torch_dtype=torch.float16)
        else:
            self.unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=torch.float16)
        self.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")
        self.scheduler.set_timesteps(self.num_inference_steps)
        self.vae.to(self.device).eval().requires_grad_(False)
        self.text_encoder.to(self.device).eval().requires_grad_(False)
        self.unet.to(self.device).eval().requires_grad_(False)

    def _encode_text(self, prompt: str) -> torch.Tensor:
        tokens = self.tokenizer(
            [prompt],
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
            return self.text_encoder(tokens.input_ids)[0]

    def _predict_noise(self, latent: torch.Tensor, t, embedding: torch.Tensor) -> torch.Tensor:
        return self.unet(latent, t, encoder_hidden_states=embedding).sample

    def _ddim_prev_step(self, noise_pred: torch.Tensor, t: int, sample: torch.Tensor) -> torch.Tensor:
        step = self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        prev_timestep = t - step
        alpha_prod_t = self.scheduler.alphas_cumprod[t]
        alpha_prod_t_prev = self.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else self.scheduler.final_alpha_cumprod
        beta_prod_t = 1 - alpha_prod_t
        pred_original_sample = (sample - beta_prod_t**0.5 * noise_pred) / alpha_prod_t**0.5
        pred_sample_direction = (1 - alpha_prod_t_prev)**0.5 * noise_pred
        return alpha_prod_t_prev**0.5 * pred_original_sample + pred_sample_direction

    def _decode_latent_to_image(self, latent: torch.Tensor) -> Image.Image:
        with torch.no_grad():
            latent = latent / self.vae.config.scaling_factor
            self.vae.to(dtype=torch.float32)
            image = self.vae.decode(latent.float()).sample
            self.vae.to(dtype=torch.float16)
        image = (image / 2 + 0.5).clamp(0, 1)
        image_np = (image[0].permute(1, 2, 0).float().cpu().numpy() * 255).round().astype(np.uint8)
        return Image.fromarray(image_np)

    def _validate_timesteps_available(self, attention_maps, timesteps, num_injection_steps: int) -> None:
        missing = [int(t) for t in timesteps[:num_injection_steps] if int(t) not in attention_maps]
        if missing:
            raise ValueError(f"attention_maps thieu timestep {missing}. Kiểm tra num_inference_steps giữa Inverter và Editor.")

    def reconstruct(
        self,
        z_T: torch.Tensor,
        null_embeddings: List[torch.Tensor],
        initial_age: int,
        gender_word: str,
    ) -> Image.Image:
        """
        Denoise từ z_T bằng prompt gốc P_alpha và danh sách null_embeddings (KHÔNG tiêm attention)
        để tái tạo lại ảnh ban đầu nhằm kiểm tra độ chính xác của Inversion.
        """
        if self.unet is None:
            self._load_models()
        p_alpha = build_prompt_alpha(initial_age, gender_word)
        cond_embedding = self._encode_text(p_alpha)
        latent = z_T.clone()
        timesteps = self.scheduler.timesteps
        with torch.no_grad():
            for i in range(self.num_inference_steps):
                t = timesteps[i]
                null_t = null_embeddings[i]
                noise_uncond = self._predict_noise(latent, t, null_t)
                noise_cond = self._predict_noise(latent, t, cond_embedding)
                noise_pred = noise_uncond + self.guidance_scale * (noise_cond - noise_uncond)
                latent = self._ddim_prev_step(noise_pred, t, latent)
        return self._decode_latent_to_image(latent)

    def edit(
        self,
        z_T: torch.Tensor,
        null_embeddings: List[torch.Tensor],
        attention_maps: Tuple[_Dict, _Dict],
        target_ages: List[int],
        gender_word: str,
        output_dir: str,
        original_image_path: Optional[str] = None,
        embedder: Optional[object] = None,
        use_local_blend: Optional[bool] = None,       # SUA: Tham số tùy chọn bật/tắt LocalBlend ở mỗi lần gọi
        local_blend_threshold: Optional[float] = None,# SUA: Ngưỡng binarize cho LocalBlend
        initial_age: Optional[int] = None,           # SUA: Độ tuổi gốc để tạo prompt reconstruction
    ) -> _Dict[int, str]:
        if self.unet is None:
            self._load_models()
        self_maps, cross_maps = attention_maps
        os.makedirs(output_dir, exist_ok=True)
        timesteps = self.scheduler.timesteps
        num_injection_steps = int(self.attention_control_ratio * self.num_inference_steps)
        self._validate_timesteps_available(self_maps, timesteps, num_injection_steps)

        # SUA: Xác định cờ use_local_blend và threshold
        do_local_blend = self.use_local_blend if use_local_blend is None else use_local_blend
        lb_threshold = self.local_blend_threshold if local_blend_threshold is None else local_blend_threshold

        results: _Dict[int, str] = {}
        for target_age in target_ages:
            p_tau = build_prompt_tau(target_age, gender_word)
            cond_embedding = self._encode_text(p_tau)
            latent = z_T.clone()

            # SUA: Chuẩn bị latent reconstruction và prompt neo cho LocalBlend
            if do_local_blend:
                recon_latent = z_T.clone()
                p_alpha = build_prompt_alpha(initial_age, gender_word) if initial_age is not None else f"photo of a {gender_word}"
                cond_embedding_recon = self._encode_text(p_alpha)
                word_inds_recon = get_word_inds(p_alpha, gender_word, self.tokenizer)
                word_inds_edit = get_word_inds(p_tau, gender_word, self.tokenizer)
                print(f"[Editor LocalBlend] Khởi chạy song song recon_latent | p_alpha='{p_alpha}' | p_tau='{p_tau[:35]}...' | threshold={lb_threshold}")

            # --- TÍNH TOÁN ATTENTION CONTROL RATIO ĐỘNG ---
            dynamic_ratio = self.attention_control_ratio
            if target_age >= 60 or target_age <= 15:
                dynamic_ratio = 0.65
            elif target_age >= 40:
                dynamic_ratio = 0.75
            else:
                dynamic_ratio = self.attention_control_ratio
            print(f"[Editor] target_age={target_age} -> dùng dynamic_attention_ratio={dynamic_ratio}")

            injector = DualAttentionInjector(
                self.unet, self_maps, cross_maps, dynamic_ratio, self.num_inference_steps
            )
            injector.register()
            try:
                for i in range(self.num_inference_steps):
                    t = timesteps[i]
                    null_t = null_embeddings[i]

                    # --- NẾU BẬT LOCALBLEND: Chạy bước denoising cho reconstruction latent song song ---
                    if do_local_blend:
                        injector.captured_cross["recon"].clear()
                        injector.captured_cross["edit"].clear()

                        with torch.no_grad():
                            noise_uncond_recon = self._predict_noise(recon_latent, t, null_t)

                        # Bật capture_mode="recon" cho conditional forward pass
                        injector.capture_mode = "recon"
                        injector.enabled = False  # Không tiêm attention vào recon pass
                        with torch.no_grad():
                            noise_cond_recon = self._predict_noise(recon_latent, t, cond_embedding_recon)
                        injector.capture_mode = None

                        with torch.no_grad():
                            noise_pred_recon = noise_uncond_recon + self.guidance_scale * (noise_cond_recon - noise_uncond_recon)
                            recon_latent = self._ddim_prev_step(noise_pred_recon, t, recon_latent)

                    # --- Denoise edit latent (tiêm attention bình thường) ---
                    with torch.no_grad():
                        noise_uncond = self._predict_noise(latent, t, null_t)

                    # Bật capture_mode="edit" cho conditional forward pass của edit
                    if do_local_blend:
                        injector.capture_mode = "edit"

                    noise_cond = injector.inject_step(
                        i, t, lambda: self._predict_noise(latent, t, cond_embedding)
                    )

                    if do_local_blend:
                        injector.capture_mode = None

                    with torch.no_grad():
                        noise_pred = noise_uncond + self.guidance_scale * (noise_cond - noise_uncond)
                        latent = self._ddim_prev_step(noise_pred, t, latent)

                        # SUA: Áp dụng LocalBlend TRỘN 2 latent ngay sau ddim_prev_step ở mỗi bước i
                        if do_local_blend:
                            if len(injector.captured_cross["recon"]) > 0 and len(injector.captured_cross["edit"]) > 0:
                                latent, _ = local_blend(
                                    recon_latent=recon_latent,
                                    edit_latent=latent,
                                    cross_attn_maps_recon=injector.captured_cross["recon"],
                                    cross_attn_maps_edit=injector.captured_cross["edit"],
                                    word_inds_recon=word_inds_recon,
                                    word_inds_edit=word_inds_edit,
                                    threshold=lb_threshold,
                                )
                            injector.captured_cross["recon"].clear()
                            injector.captured_cross["edit"].clear()

                        if torch.isnan(latent).any():
                            raise RuntimeError(f"[Editor] NaN xuat hien tai step i={i}, t={int(t)}, target_age={target_age}.")
            finally:
                injector.restore()

            image = self._decode_latent_to_image(latent)

            # --- ÁP DỤNG MASK-BASED BLENDING ---
            # Nếu truyền đường dẫn ảnh gốc vào hàm edit, tiến hành giữ phông nền, tóc và áo quần cũ
            if original_image_path and os.path.exists(original_image_path) and embedder is not None:
                image = apply_mask_blending(original_image_path, image, embedder)

            path = os.path.join(output_dir, f"age_{target_age}.png")
            image.save(path)
            results[target_age] = path
            print(f"[Editor] target_age={target_age} -> {path}")
        return results


## Module 4 — InsightFace Embedding (tương đương `src/search/embedding.py`)

In [ ]:
import glob

import cv2

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")


class FaceEmbedder:
    """Boc insightface.app.FaceAnalysis: trich xuat embedding 512-chieu cho 1 anh, va build
    gallery embedding cho toan bo anh trong 1 folder."""

    def __init__(
        self,
        model_name: str = "buffalo_l",
        ctx_id: int = -1,
        det_size: Tuple[int, int] = (256, 256),
    ):
        """Luu config (ctx_id: -1=CPU mac dinh de tranh tranh chap VRAM voi cac module
        diffusion). Model CHUA duoc load o day, se load lazily trong _load_model()."""
        self.model_name = model_name
        self.ctx_id = ctx_id
        self.det_size = det_size
        self.app = None

    def _load_model(self) -> None:
        """Load FaceAnalysis(buffalo_l) va prepare() theo dung ctx_id/det_size."""
        from insightface.app import FaceAnalysis

        self.app = FaceAnalysis(name=self.model_name)
        self.app.prepare(ctx_id=self.ctx_id, det_size=self.det_size)

    def embed(self, image_path: str) -> np.ndarray:
        """Doc anh bang cv2.imread (BGR - dung chuan insightface, KHONG dung PIL vi PIL doc
        RGB se cho embedding sai lech ma khong bao loi), tra ve normed_embedding (512-dim).
        Neu khong detect duoc mat, raise ValueError ro rang (khong tra ve None/vector rong)."""
        if self.app is None:
            self._load_model()

        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Khong doc duoc anh: {image_path}")

        faces = self.app.get(img)
        if len(faces) == 0:
            raise ValueError(f"Khong phat hien duoc khuon mat nao trong anh: {image_path}")

        return faces[0].normed_embedding

    def detect_faces(self, image_path: str) -> list:
        """SUA (moi them): tra ve list cac mat detect duoc (bbox, kps 5 diem, det_score).
        kps dung de align_to_ffhq() - QUAN TRONG: bi thieu tu dau, khien vong lap FG-NET
        chua tung align anh, dan den ket qua co the thap hon nang luc that cua he thong."""
        if self.app is None:
            self._load_model()
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Khong doc duoc anh: {image_path}")
        faces = self.app.get(img)
        # SUA: them "gender" (0=nu, 1=nam, InsightFace tra ve san, khong ton gi them)
        # - dung cho Enhanced Prompts (EP), da ghi chu la quan trong nhung CHUA tung
        # duoc ap dung trong vong lap danh gia FG-NET (luon dung "person" co dinh).
        return [{"bbox": f.bbox, "kps": f.kps, "det_score": f.det_score,
                  "gender": int(f.gender)} for f in faces]

    def build_gallery(self, folder_path: str) -> Tuple[List[np.ndarray], List[str], List[str]]:
        """Chay embedding cho toan bo anh trong folder_path. Anh loi KHONG lam dung ca ham,
        nhung cung KHONG am tham bo qua: gom vao failed_files. Tra ve
        (embeddings, identity_labels, failed_files)."""
        image_paths = sorted(
            f
            for f in glob.glob(os.path.join(folder_path, "*"))
            if f.lower().endswith(IMAGE_EXTENSIONS)
        )

        embeddings: List[np.ndarray] = []
        identity_labels: List[str] = []
        failed_files: List[str] = []

        for image_path in image_paths:
            try:
                embedding = self.embed(image_path)
            except ValueError as e:
                print(f"[FaceEmbedder] BO QUA anh loi: {e}")
                failed_files.append(image_path)
                continue
            embeddings.append(embedding)
            identity_labels.append(os.path.splitext(os.path.basename(image_path))[0])

        if failed_files:
            print(
                f"[FaceEmbedder] Gallery '{folder_path}' thieu {len(failed_files)}/"
                f"{len(image_paths)} identity do loi doc/detect anh."
            )

        return embeddings, identity_labels, failed_files

## Tiền xử lý & Hậu xử lý khuôn mặt (Preprocessing, Face Alignment & Mask-based Blending)

1. **Pipeline Tiền xử lý (Được kiểm chứng từ `test_preprocessing_fgnet.ipynb`):**
   - **`apply_adaptive_padding`:** Dùng `cv2.BORDER_REPLICATE` thay vì reflect để triệt tiêu hoa văn hình thoi khi mặt sát mép ảnh.
   - **`apply_white_balance`:** Thuật toán **Shades of Gray (Minkowski p-norm, $p=6$, Finlayson & Trezzi 2004)** kèm dynamic alpha-blending ($\Delta_{\max} > 35.0$), khử vệt xanh trán ở specular highlight và bảo toàn sắc da ấm tự nhiên.
   - **`run_codeformer`:** Phục hồi chi tiết khuôn mặt với fidelity weight duy nhất $w=0.7$ (Fidelity Focus) qua CodeFormer CLI `--face_upsample`.
2. **Face Alignment:** Căn chỉnh khuôn mặt về chuẩn 512x512 dựa trên landmarks InsightFace (chuẩn NVIDIA FFHQ).
3. **Mask-based Blending:** Hậu xử lý pha trộn vùng mặt đã sinh vào ảnh gốc bằng soft mask để bảo toàn phông nền, tóc và áo quần gốc.


In [ ]:
from PIL import Image
import numpy as np
import cv2
import os
import subprocess
import sys
import shutil
from typing import Tuple, List, Dict, Optional
from scipy.ndimage import gaussian_filter

# ==============================================================================
# 1. PIPELINE TIỀN XỬ LÝ (ADAPTIVE PADDING + SHADES OF GRAY WB + CODEFORMER w=0.7)
# ==============================================================================

def apply_adaptive_padding(
    image_rgb: np.ndarray, 
    face_occupancy_thresh: float = 0.85, 
    pad_ratio: float = 0.20,
    border_mode: str = "replicate",
    embedder = None
) -> Tuple[np.ndarray, bool]:
    """
    Bước 1: Adaptive Padding với cv2.BORDER_REPLICATE.
    Nếu khuôn mặt chiếm >= 85% chiều dài/rộng ảnh gốc hoặc sát mép (< 5% biên),
    thêm padding lặp mép 20% mỗi cạnh. Dùng BORDER_REPLICATE để tránh hoa văn hình thoi do phản chiếu mép mặt.
    """
    H, W = image_rgb.shape[:2]
    faces = []
    if embedder is not None and hasattr(embedder, 'detect_faces'):
        try:
            faces = embedder.detect_faces(image_rgb)
        except Exception:
            pass

    if len(faces) == 0:
        try:
            cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
            face_cascade = cv2.CascadeClassifier(cascade_path)
            gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
            h_faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(30, 30))
            for (x, y, w, h) in h_faces:
                faces.append({"bbox": [x, y, x + w, y + h]})
        except Exception:
            pass

    need_padding = False
    if len(faces) > 0:
        face = max(faces, key=lambda f: (f["bbox"][2] - f["bbox"][0]) * (f["bbox"][3] - f["bbox"][1]))
        x1, y1, x2, y2 = face["bbox"]
        w_face, h_face = x2 - x1, y2 - y1
        occ_w = w_face / W
        occ_h = h_face / H
        if occ_w >= face_occupancy_thresh or occ_h >= face_occupancy_thresh or x1 < 0.05 * W or x2 > 0.95 * W or y1 < 0.05 * H or y2 > 0.95 * H:
            need_padding = True
    else:
        if min(H, W) < 300:
            need_padding = True

    if need_padding:
        pad_h = int(H * pad_ratio)
        pad_w = int(W * pad_ratio)
        cv_border = cv2.BORDER_REPLICATE if border_mode == "replicate" else cv2.BORDER_REFLECT_101
        padded = cv2.copyMakeBorder(image_rgb, pad_h, pad_h, pad_w, pad_w, cv_border)
        return padded, True
    return image_rgb, False


def apply_white_balance(
    image_rgb: np.ndarray,
    p: int = 6,                      # Minkowski p-norm (Shades of Gray, Finlayson & Trezzi 2004)
    max_shift_thresh: float = 35.0,  # Ngưỡng lệch kênh màu tối đa (30-40 đơn vị)
    gain_min: float = 0.75,          # Giới hạn giảm kênh tối đa 25%
    gain_max: float = 1.30,          # Giới hạn tăng kênh tối đa 30%
    return_info: bool = False
) -> np.ndarray:
    """
    Bước 2: Shades of Gray White Balance (Minkowski p-norm, p=6, Finlayson & Trezzi 2004)
    kết hợp giới hạn gain [0.75, 1.30] & Dynamic Alpha Blending.
    Trọng số dồn về vùng sáng nhất (specular highlights), triệt tiêu vệt xanh lá ở trán
    trên ảnh sepia đậm (003A35.JPG, 004A37.JPG) và bảo toàn sắc da ấm tự nhiên.
    """
    # 1. Chuyển sang float64 và chuẩn hóa về [0, 1] trước khi tính pixel^p tránh tràn số
    img_norm = image_rgb.astype(np.float64) / 255.0

    # 2. Tính Minkowski p-norm (p=6) cho từng kênh: avg_c = (1/N * sum(pixel^p))^(1/p)
    norm_r = np.power(np.mean(np.power(img_norm[:, :, 0], p)), 1.0 / p)
    norm_g = np.power(np.mean(np.power(img_norm[:, :, 1], p)), 1.0 / p)
    norm_b = np.power(np.mean(np.power(img_norm[:, :, 2], p)), 1.0 / p)

    avg_gray = (norm_r + norm_g + norm_b) / 3.0

    # 3. Tính raw gains
    raw_gain_r = float(avg_gray / (norm_r + 1e-8))
    raw_gain_g = float(avg_gray / (norm_g + 1e-8))
    raw_gain_b = float(avg_gray / (norm_b + 1e-8))

    # 4. Kẹp gains vào khoảng an toàn [gain_min, gain_max]
    clamped_gain_r = float(np.clip(raw_gain_r, gain_min, gain_max))
    clamped_gain_g = float(np.clip(raw_gain_g, gain_min, gain_max))
    clamped_gain_b = float(np.clip(raw_gain_b, gain_min, gain_max))

    # 5. Áp dụng gains đã kẹp lên ảnh gốc float32
    img_float = image_rgb.astype(np.float32)
    wb_temp = np.zeros_like(img_float)
    wb_temp[:, :, 0] = np.clip(img_float[:, :, 0] * clamped_gain_r, 0, 255)
    wb_temp[:, :, 1] = np.clip(img_float[:, :, 1] * clamped_gain_g, 0, 255)
    wb_temp[:, :, 2] = np.clip(img_float[:, :, 2] * clamped_gain_b, 0, 255)

    # 6. Đo độ lệch màu thực tế trước và sau WB
    avg_r = float(np.mean(img_float[:, :, 0]))
    avg_g = float(np.mean(img_float[:, :, 1]))
    avg_b = float(np.mean(img_float[:, :, 2]))

    shift_r = abs(float(np.mean(wb_temp[:, :, 0])) - avg_r)
    shift_g = abs(float(np.mean(wb_temp[:, :, 1])) - avg_g)
    shift_b = abs(float(np.mean(wb_temp[:, :, 2])) - avg_b)
    max_shift = max(shift_r, shift_g, shift_b)

    # 7. Dynamic Alpha Blending nếu max_shift > max_shift_thresh
    if max_shift > max_shift_thresh:
        alpha = max_shift_thresh / (max_shift + 1e-6)
    else:
        alpha = 1.0

    blended = img_float * (1.0 - alpha) + wb_temp * alpha
    out_rgb = np.clip(blended, 0, 255).astype(np.uint8)

    info = {
        "p_norm": (round(float(norm_r), 4), round(float(norm_g), 4), round(float(norm_b), 4)),
        "avg_orig": (round(avg_r, 1), round(avg_g, 1), round(avg_b, 1)),
        "raw_gains": (round(raw_gain_r, 3), round(raw_gain_g, 3), round(raw_gain_b, 3)),
        "clamped_gains": (round(clamped_gain_r, 3), round(clamped_gain_g, 3), round(clamped_gain_b, 3)),
        "max_shift": round(max_shift, 1),
        "alpha": round(alpha, 2),
        "avg_out": (round(float(out_rgb[:, :, 0].mean()), 1),
                    round(float(out_rgb[:, :, 1].mean()), 1),
                    round(float(out_rgb[:, :, 2].mean()), 1))
    }
    if return_info:
        return out_rgb, info
    return out_rgb


def run_codeformer(
    image_rgb: np.ndarray,
    fidelity_weight: float = 0.7,
    unique_tag: str = "temp",
    temp_dir: str = "/kaggle/working/temp_codeformer"
) -> np.ndarray:
    """
    Bước 3: Chạy CodeFormer inference trên 1 ảnh RGB với fidelity_weight cho trước (mặc định w=0.7).
    Gọi CLI inference_codeformer.py --face_upsample như đã validate trên tập mẫu.
    Trả về ảnh RGB sau phục hồi (np.ndarray uint8). Raise Exception nếu thất bại để kích hoạt fallback.
    """
    codeformer_dir = globals().get("CODEFORMER_DIR", "/kaggle/working/CodeFormer")
    codeformer_script = os.path.join(codeformer_dir, "inference_codeformer.py")
    if not os.path.exists(codeformer_script):
        for alt_d in ["/kaggle/working/CodeFormer", "./CodeFormer", "../CodeFormer"]:
            cand = os.path.join(alt_d, "inference_codeformer.py")
            if os.path.exists(cand):
                codeformer_script = cand
                break

    if not os.path.exists(codeformer_script):
        raise FileNotFoundError(f"Không tìm thấy script CodeFormer tại: {codeformer_script}")

    in_dir = os.path.join(temp_dir, f"in_{unique_tag}")
    out_dir = os.path.join(temp_dir, f"out_{unique_tag}")
    os.makedirs(in_dir, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)

    in_file = os.path.join(in_dir, "input.png")
    cv2.imwrite(in_file, cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

    cmd = [
        sys.executable, codeformer_script,
        "-w", str(fidelity_weight),
        "--input_path", in_file,
        "-o", out_dir,
        "--face_upsample"
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
    if proc.returncode != 0:
        raise RuntimeError(f"CodeFormer CLI lỗi (code {proc.returncode}): {proc.stderr[:200]}")

    res_path = os.path.join(out_dir, "final_results", "input.png")
    if not os.path.exists(res_path):
        fin_dir = os.path.join(out_dir, "final_results")
        if os.path.exists(fin_dir) and os.listdir(fin_dir):
            res_path = os.path.join(fin_dir, os.listdir(fin_dir)[0])
        else:
            raise FileNotFoundError(f"Không tìm thấy kết quả CodeFormer tại: {out_dir}")

    res_bgr = cv2.imread(res_path)
    if res_bgr is None:
        raise ValueError(f"Không đọc được ảnh kết quả CodeFormer: {res_path}")

    # Dọn dẹp thư mục tạm để tiết kiệm dung lượng đĩa
    try:
        shutil.rmtree(in_dir, ignore_errors=True)
        shutil.rmtree(out_dir, ignore_errors=True)
    except Exception:
        pass

    return cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)

# ==============================================================================
# 2. CĂN CHỈNH KHUÔN MẶT & HẬU XỬ LÝ BLENDING
# ==============================================================================

def align_to_ffhq(image_path: str, kps: np.ndarray, output_size: int = 512) -> Image.Image:
    """Align mot anh theo dung cong thuc goc NVIDIA (ffhq_dataset/face_alignment.py),
    dung 4 diem tu 5-point landmarks InsightFace (bo qua diem mui):
    mat trai, mat phai, khoe mieng trai, khoe mieng phai.
    Tich hop pad-and-blur (NVIDIA NVlabs) khi quad vuot bien de triet tieu vien den/hoa van hinh thoi."""
    eye_left = np.array(kps[0], dtype=np.float64)
    eye_right = np.array(kps[1], dtype=np.float64)
    mouth_left = np.array(kps[3], dtype=np.float64)
    mouth_right = np.array(kps[4], dtype=np.float64)

    eye_avg = (eye_left + eye_right) * 0.5
    eye_to_eye = eye_right - eye_left
    mouth_avg = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg

    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    qsize = max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    x *= qsize
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])

    # Đọc ảnh (hỗ trợ cả str filepath, PIL.Image hoặc numpy ndarray)
    if isinstance(image_path, str):
        img = Image.open(image_path).convert("RGB")
    elif isinstance(image_path, Image.Image):
        img = image_path.convert("RGB")
    elif isinstance(image_path, np.ndarray):
        img = Image.fromarray(image_path).convert("RGB")
    else:
        raise ValueError(f"Unsupported image type: {type(image_path)}")
    img_w, img_h = img.size

    # ===== MỚI: Tính padding cần thiết nếu quad vượt biên (Thuật toán chuẩn FFHQ / NVlabs) =====
    border = max(int(round(qsize * 0.1)), 3)
    pad = (
        max(int(-np.floor(quad[:, 0].min())) + border, 0),
        max(int(-np.floor(quad[:, 1].min())) + border, 0),
        max(int(np.ceil(quad[:, 0].max())) - img_w + border, 0),
        max(int(np.ceil(quad[:, 1].max())) - img_h + border, 0)
    )

    if max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        pad_left, pad_top, pad_right, pad_bottom = pad

        # Vượt biên đáng kể -> mở rộng canvas bằng phản chiếu + làm mờ vùng nối
        img_np = np.pad(
            np.float32(np.array(img)),
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="reflect",
        )
        h, w, _ = img_np.shape
        y_grid, x_grid, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(
            1.0 - np.minimum(np.float32(x_grid) / pad_left, np.float32(w - 1 - x_grid) / pad_right),
            1.0 - np.minimum(np.float32(y_grid) / pad_top, np.float32(h - 1 - y_grid) / pad_bottom),
        )
        blur_sigma = qsize * 0.02
        img_np += (gaussian_filter(img_np, [blur_sigma, blur_sigma, 0]) - img_np) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img_np += (np.median(img_np, axis=(0, 1)) - img_np) * np.clip(mask, 0.0, 1.0)
        img = Image.fromarray(np.uint8(np.clip(np.rint(img_np), 0, 255)))
        # Dời quad theo đúng offset padding vừa thêm
        quad = quad + np.array([pad_left, pad_top])

    img = img.transform((output_size, output_size), Image.QUAD, (quad + 0.5).flatten(), Image.BILINEAR)
    return img


def align_image_for_pipeline(image_path: str, embedder, output_dir: str, unique_tag: str) -> str:
    """Ham tien ich: detect mat, align, LUU ra file voi TEN DUY NHAT, tra ve duong dan
    de dua vao inverter.invert(). Raise ValueError ro rang neu khong detect duoc mat."""
    faces = embedder.detect_faces(image_path)
    if len(faces) == 0:
        raise ValueError(f"Khong phat hien khuon mat de align: {image_path}")
    aligned = align_to_ffhq(image_path, faces[0]["kps"])
    os.makedirs(output_dir, exist_ok=True)
    aligned_path = os.path.join(output_dir, f"aligned_{unique_tag}.png")
    aligned.save(aligned_path)
    return aligned_path


def apply_mask_blending(original_image_path: str, generated_image: Image.Image, embedder) -> Image.Image:
    """
    Hậu xử lý Mask-based Blending: Sử dụng landmark của InsightFace để tạo mặt nạ mềm (soft mask)
    cho vùng khuôn mặt, sau đó pha trộn ảnh đã edit vào ảnh gốc để giữ trọn phông nền, tóc và áo quần.
    """
    orig_img = cv2.imread(original_image_path)
    if orig_img is None:
        return generated_image
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    
    gen_np = np.array(generated_image.resize((orig_img.shape[1], orig_img.shape[0]), Image.BICUBIC))
    
    faces = embedder.detect_faces(original_image_path)
    if len(faces) == 0:
        return generated_image
        
    face = faces[0]
    kps = face["kps"]
    
    mask = np.zeros(orig_img.shape[:2], dtype=np.float32)
    bbox = face["bbox"].astype(int)
    x1, y1, x2, y2 = bbox
    w, h = x2 - x1, y2 - y1
    
    x1 = max(0, int(x1 - 0.1 * w))
    x2 = min(orig_img.shape[1], int(x2 + 0.1 * w))
    y1 = max(0, int(y1 - 0.2 * h))
    y2 = min(orig_img.shape[0], int(y2 + 0.1 * h))
    
    center = ((x1 + x2) // 2, (y1 + y2) // 2)
    axes = (int((x2 - x1) * 0.55), int((y2 - y1) * 0.6))
    cv2.ellipse(mask, center, axes, 0, 0, 360, 1.0, -1)
    
    blur_kernel = int(min(w, h) * 0.25)
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    mask = cv2.GaussianBlur(mask, (blur_kernel, blur_kernel), 0)
    mask_3ch = np.stack([mask, mask, mask], axis=-1)
    
    blended = orig_img.astype(np.float32) * (1.0 - mask_3ch) + gen_np.astype(np.float32) * mask_3ch
    blended = np.clip(blended, 0, 255).astype(np.uint8)
    return Image.fromarray(blended)


## Khởi tạo Pipeline Models (Inverter, Editor, Embedder — Khởi tạo 1 lần duy nhất)

Nạp sẵn UNet đã fine-tune, SD v1.5 components, và InsightFace vào bộ nhớ. Toàn bộ các cell Demo và Đánh giá FG-NET bên dưới sẽ dùng chung instance này, không khởi tạo lại để tiết kiệm thời gian và bộ nhớ.


In [ ]:
ckpt_dir = config["paths"]["specialized_unet_ckpt"]
print("Khởi tạo Inverter, Editor và Embedder dùng chung cho toàn bộ pipeline...")
inverter = NullTextInverter(
    pretrained_model_name_or_path=config["base_model"]["pretrained_model_name_or_path"],
    unet_checkpoint_dir=ckpt_dir,
    num_inference_steps=config["inversion"]["num_inference_steps"],
    guidance_scale=config["inversion"]["guidance_scale"],
    num_inner_steps=config["inversion"]["num_inner_steps"],
    early_stop_epsilon=float(config["inversion"]["early_stop_epsilon"]),
    image_size=config["inversion"]["image_size"],
)
editor = Editor(
    pretrained_model_name_or_path=config["base_model"]["pretrained_model_name_or_path"],
    unet_checkpoint_dir=ckpt_dir,
    num_inference_steps=config["inversion"]["num_inference_steps"],
    guidance_scale=config["editing"]["guidance_scale"],
    attention_control_ratio=config["editing"]["attention_control_ratio"],
    image_size=config["editing"]["image_size"],
)
embedder = FaceEmbedder(
    model_name=config["embedding"]["model_name"],
    ctx_id=config["embedding"]["ctx_id"],
    det_size=tuple(config["embedding"]["det_size"]),
)
print("✅ Đã sẵn sàng: inverter, editor, embedder.")


## Thử nghiệm ảnh đơn (Single Image Demo)


In [ ]:
import pandas as pd
from PIL import Image
# Chọn ảnh test
TEST_IMAGE_NAME = "01366.png"
TEST_IMAGE_PATH = os.path.join(FFHQ_DIR, TEST_IMAGE_NAME)
# Tra đúng label thật của ảnh này từ CSV - KHÔNG hardcode đoán tuổi/giới tính nữa
df = pd.read_csv(LABELS_CSV)
image_number = int(TEST_IMAGE_NAME.replace(".png", ""))
row = df[df["image_number"] == image_number].iloc[0]
INITIAL_AGE = age_group_to_age(row["age_group"])
GENDER_WORD = gender_to_word(row["gender"], INITIAL_AGE)
TARGET_AGES = [20, 30, 40, 50, 60, 70, 80]   # mục tiêu tự chọn tay, không liên quan label gốc
print(f"Ảnh: {TEST_IMAGE_NAME} | age_group thật: {row['age_group']} | gender thật: {row['gender']}")
print(f"=> INITIAL_AGE={INITIAL_AGE}, GENDER_WORD='{GENDER_WORD}'")
# Căn chỉnh ảnh test trước khi đưa vào Invert
test_aligned_dir = os.path.join(OUTPUT_DIR, "test_aligned")
ALIGNED_TEST_IMAGE_PATH = align_image_for_pipeline(
    TEST_IMAGE_PATH, embedder, test_aligned_dir, "demo_01366"
)
print(f"✅ Đã căn chỉnh ảnh test theo chuẩn FFHQ 512x512: {ALIGNED_TEST_IMAGE_PATH}")
print("\nẢnh input test (đã align 512x512):")
display(Image.open(ALIGNED_TEST_IMAGE_PATH))


### Chạy Null-text Inversion (Module 2)


In [ ]:
print("Chay Module 2 (Null-text Inversion)...")
# Chạy Inversion trên ảnh đã căn chỉnh 512x512
z_T, null_embeddings, (self_maps, cross_maps) = inverter.invert(
    ALIGNED_TEST_IMAGE_PATH, INITIAL_AGE, GENDER_WORD
)
# Kiểm tra NaN cho cả Self và Cross Attention maps
_nan_found_attn = False
for maps_dict, tag in [(self_maps, "self_maps"), (cross_maps, "cross_maps")]:
    for _t, _layers in maps_dict.items():
        for _name, _tensor in _layers.items():
            if torch.isnan(_tensor).any():
                print(f"  ❌ {tag}[t={_t}][{_name}] CO NaN!")
                _nan_found_attn = True
if not _nan_found_attn:
    print("  ✅ Khong co NaN trong ca self_maps va cross_maps")
print("  ✅ z_T binh thuong" if not torch.isnan(z_T).any() else "  ❌ z_T CO NaN!")
# Giữ inverter trong bộ nhớ cho các tác vụ tiếp theo
torch.cuda.empty_cache()
print("z_T shape:", z_T.shape)
print("so luong null_t:", len(null_embeddings))
print(f"so timestep: self_maps={len(self_maps)}, cross_maps={len(cross_maps)}")
attention_maps = (self_maps, cross_maps)


In [ ]:
# ============================================================================
# CELL CHẨN ĐOÁN TẠM THỜI — kiểm tra z_T và null_embeddings có bất thường không
# (NaN/Inf/giá trị vượt ngưỡng bình thường của latent SD 1.5, thường trong khoảng
# [-5, 5]). Nếu số liệu bình thường ở đây mà ảnh vẫn hỏng -> lỗi nằm ở bước
# generate/decode, không phải ở bản thân z_T/null_embeddings.
# ============================================================================
print("--- z_T ---")
print(f"shape={z_T.shape}, dtype={z_T.dtype}")
print(f"min={z_T.min().item():.4f}, max={z_T.max().item():.4f}, mean={z_T.mean().item():.4f}, std={z_T.std().item():.4f}")
print(f"có NaN: {torch.isnan(z_T).any().item()} | có Inf: {torch.isinf(z_T).any().item()}")

print("\n--- null_embeddings (tất cả các bước) ---")
all_null = torch.cat(null_embeddings, dim=0)
print(f"số bước: {len(null_embeddings)}, shape mỗi bước: {null_embeddings[0].shape}")
print(f"min={all_null.min().item():.4f}, max={all_null.max().item():.4f}, mean={all_null.mean().item():.4f}, std={all_null.std().item():.4f}")
print(f"có NaN: {torch.isnan(all_null).any().item()} | có Inf: {torch.isinf(all_null).any().item()}")

print("\n--- So sánh giá trị null_embeddings bước đầu vs bước cuối ---")
print(f"Bước 0  : min={null_embeddings[0].min().item():.4f}, max={null_embeddings[0].max().item():.4f}")
print(f"Bước cuối: min={null_embeddings[-1].min().item():.4f}, max={null_embeddings[-1].max().item():.4f}")

print("\n--- Kiểm tra checkpoint UNet đang dùng ---")
print(f"CKPT_DIR = {CKPT_DIR}")
print(f"unet_checkpoint_dir thực tế của inverter: {inverter.unet_checkpoint_dir}")
if inverter.unet_checkpoint_dir:
    ckpt_files = os.listdir(inverter.unet_checkpoint_dir)
    print(f"Số file trong checkpoint dir: {len(ckpt_files)}")
    print(f"Danh sách: {ckpt_files[:10]}")
    # Kiểm tra kích thước file .safetensors/.bin chính có bất thường (quá nhỏ = hỏng/tải dở) không
    for fname in ckpt_files:
        if fname.endswith(('.safetensors', '.bin')):
            fpath = os.path.join(inverter.unet_checkpoint_dir, fname)
            size_mb = os.path.getsize(fpath) / (1024*1024)
            print(f"  {fname}: {size_mb:.1f} MB" + ("  ⚠️ QUÁ NHỎ, NGHI NGỜ FILE HỎNG/TẢI DỞ" if size_mb < 100 else ""))

print("\n" + "="*70)
print("ĐỌC KẾT QUẢ:")
print("- Nếu có NaN/Inf ở z_T hoặc null_embeddings -> lỗi số học trong quá trình")
print("  invert (nghi ngờ fp16 overflow) -> báo lại NGAY, không cần làm bước B.")
print("- Nếu min/max của z_T hoặc null_embeddings vượt xa [-10, 10] -> bất thường,")
print("  khả năng cao checkpoint hỏng hoặc mismatch kiến trúc.")
print("- Nếu file .safetensors/.bin có cảnh báo 'QUÁ NHỎ' -> checkpoint tải dở/hỏng,")
print("  đây nhiều khả năng CHÍNH LÀ nguyên nhân -> tải lại checkpoint, không cần bước B.")
print("- Nếu mọi số liệu đều bình thường -> chuyển sang Bước B (test CKPT_DIR=None).")
print("="*70)

In [ ]:
# ==============================================================================
# BƯỚC BẮT BUỘC: KIỂM TRA TÁI TẠO (RECONSTRUCTION SANITY-CHECK)
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
print("Đang chạy kiểm tra tái tạo ảnh gốc từ z_T và null_embeddings...")
# 1. Tái tạo ảnh bằng prompt gốc P_alpha
reconstructed_img = editor.reconstruct(
    z_T=z_T,
    null_embeddings=null_embeddings,
    initial_age=INITIAL_AGE,
    gender_word=GENDER_WORD,
)
# 2. Lưu ảnh tạm thời để đưa qua FaceEmbedder đo ID Score
temp_rec_path = os.path.join(OUTPUT_DIR, "sanity_check_reconstruction.png")
reconstructed_img.save(temp_rec_path)
# 3. Đo độ tương đồng nhận diện giữa ảnh gốc (đã align) và ảnh tái tạo
orig_emb = embedder.embed(ALIGNED_TEST_IMAGE_PATH)
rec_emb = embedder.embed(temp_rec_path)
rec_id_score = float(np.dot(orig_emb, rec_emb))
# 4. Hiển thị đối chiếu trực quan 2 ảnh
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
orig_display = Image.open(ALIGNED_TEST_IMAGE_PATH).resize((512, 512))
axes[0].imshow(orig_display)
axes[0].set_title(f"Ảnh đầu vào đã align (Tuổi: {INITIAL_AGE})", fontsize=12)
axes[0].axis("off")
axes[1].imshow(reconstructed_img)
axes[1].set_title(f"Ảnh tái tạo từ Inversion\nID Score: {rec_id_score:.4f}", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()
# 5. Đánh giá chất lượng và cảnh báo
print(f"-> Cosine Similarity (ID Score): {rec_id_score:.4f}")
if rec_id_score >= 0.90:
    print("✅ XÁC THỰC THÀNH CÔNG: Inversion bảo toàn xuất sắc đặc trưng khuôn mặt.")
elif rec_id_score >= 0.75:
    print("⚠️ CẢNH BÁO: Tái tạo ở mức chấp nhận được nhưng có thể mất chi tiết nhỏ. Nên tăng num_inner_steps lên 15 nếu ảnh edit bị méo.")
else:
    print("❌ THẤT BÀI: Inversion bị lệch cấu trúc nghiêm trọng (ID Score quá thấp).")
    print("   KHUYẾN NGHỊ: Không chạy tiếp Module 3. Cần kiểm tra lại:")
    print("   - Đảm bảo ảnh test đã được align_to_ffhq đúng chuẩn 512x512.")
    print("   - Tăng num_inner_steps trong config['inversion'] từ 10 lên 15 hoặc 20.")
    print("   - Đảm bảo optimizer uncond_embeddings được giữ ở FP32 khi tính gradient.")
